# Treino do detector de avarias (YOLO)

Passo a passo, uma célula por vez (Shift+Enter). Antes de começar: **Ambiente de execução > Alterar tipo de ambiente de execução > GPU T4**.

O que este caderno faz: treina um YOLOv8 nano com os recortes de um `.zip`, mostra as métricas e exporta o modelo em ONNX para rodar no navegador.

Dois conjuntos de treino, para comparar:
- `yolo-A-pinturas.zip`: só as pinturas do ArtInsight (perda de tinta marcada por especialistas, mais as marcações da autora).
- `yolo-B-pinturas-steelbook.zip`: as mesmas pinturas mais os recortes do steelbook.

Rode o caderno inteiro uma vez para cada zip (Ambiente de execução > Reiniciar sessão entre uma rodada e outra). O nome do zip vira o nome do treino e do arquivo `.onnx`, então um resultado não sobrescreve o outro.

In [ ]:
# 1. Instalar o YOLO (Ultralytics) e conferir a GPU
!pip -q install ultralytics onnx onnxruntime onnxslim
!nvidia-smi -L

## 2. Enviar o zip do dataset
A célula abaixo abre um botão de upload. Escolha `dataset/yolo-A-pinturas.zip` ou `dataset/yolo-B-pinturas-steelbook.zip` da pasta do projeto.

In [ ]:
from google.colab import files
import zipfile, pathlib, shutil

enviados = files.upload()
nome_zip = next(iter(enviados))
nome_treino = pathlib.Path(nome_zip).stem.replace(' ', '_').split('(')[0].rstrip('_')   # ex.: yolo-B-pinturas-steelbook
destino = pathlib.Path('/content/yolo')
shutil.rmtree(destino, ignore_errors=True)
with zipfile.ZipFile(nome_zip) as z:
    z.extractall(destino)

# data.yaml com caminho absoluto (o Colab nao resolve o caminho relativo do arquivo original)
(destino / 'data.yaml').write_text(
    'path: /content/yolo\ntrain: images/train\nval: images/val\nnames:\n  0: avaria\n'
)
print('treino:', nome_treino)
for conjunto in ('train', 'val'):
    quantidade = len(list((destino / 'images' / conjunto).glob('*.jpg')))
    print(conjunto, ':', quantidade, 'recortes')

## 3. Treinar
YOLOv8 nano, partindo de pesos pré-treinados (transferência de aprendizado). Leva uns 10 a 20 minutos na GPU T4.

Escolhas pensadas para poucos dados e para arte plana: giros e espelhamentos (a mancha não tem "lado certo"), variação de brilho e saturação moderada (a cor da ferrugem importa, então não exagero) e variação de escala (as fotos de detalhe e a geral têm tamanhos diferentes).

In [ ]:
from ultralytics import YOLO

modelo = YOLO('yolov8n.pt')
resultado = modelo.train(
    data='/content/yolo/data.yaml',
    imgsz=640, epochs=100, batch=16, patience=30, seed=7,
    flipud=0.5, fliplr=0.5, degrees=0,
    hsv_h=0.01, hsv_s=0.4, hsv_v=0.4,
    scale=0.5, mosaic=1.0,
    project='/content/treino', name=nome_treino, exist_ok=True,
)

## 4. Ver as métricas
- **precision**: das caixas que o modelo desenhou, quantas eram dano de verdade.
- **recall**: dos danos que você marcou, quantos o modelo achou.
- **mAP50**: resumo dos dois, com caixa "aceita" se sobrepõe pelo menos metade ao rótulo.

Atenção: a validação é feita com poucas imagens (uma pintura, e no conjunto B também duas fotos do steelbook), então o número oscila muito e não vale como medida firme. O teste que vale é nas pinturas e nos álbuns que o modelo nunca viu.

In [ ]:
metricas = modelo.val()
print('precision %.2f | recall %.2f | mAP50 %.2f' % (metricas.box.mp, metricas.box.mr, metricas.box.map50))

from IPython.display import Image, display
for arquivo in ('results.png', 'val_batch0_pred.jpg'):
    caminho = pathlib.Path('/content/treino') / nome_treino / arquivo
    if caminho.exists():
        display(Image(filename=str(caminho)))

## 5. Exportar para o navegador (ONNX)
Gera `avaria.onnx`, que vai para a pasta `models/` do projeto. O tamanho de entrada é fixo em 640 x 640.

In [ ]:
melhor = YOLO('/content/treino/' + nome_treino + '/weights/best.pt')
caminho_onnx = melhor.export(format='onnx', imgsz=640, opset=12, simplify=True, dynamic=False)
arquivo_onnx = '/content/' + nome_treino + '.onnx'
shutil.copy(caminho_onnx, arquivo_onnx)
print('tamanho: %.1f MB' % (pathlib.Path(arquivo_onnx).stat().st_size / 1048576))
files.download(arquivo_onnx)

## 6. Teste em peças novas (os álbuns)
Envie fotos que **não** entraram no treino. O caderno reduz cada foto para 2560 px no lado maior (a escala de treino), corta em recortes de 640 px, roda o modelo em cada um e junta as caixas. É exatamente o que o app fará no celular. Saída: as fotos com as caixas desenhadas.

Se você rotular essas fotos no `rotular.html`, dá para calcular precision e recall também nelas. Mesmo sem rótulos, olhar as caixas já mostra o essencial: o modelo marca desgaste real ou marca a arte impressa?

In [ ]:
import torch, torchvision
from PIL import Image as PImage, ImageOps, ImageDraw

LADO = 2560; TAM = 640; PASSO = 480

def prever_em_recortes(modelo, imagem, confianca=0.25):
    imagem = ImageOps.exif_transpose(imagem).convert('RGB')
    fator = LADO / max(imagem.size)
    if fator < 1:
        imagem = imagem.resize((round(imagem.width * fator), round(imagem.height * fator)), PImage.LANCZOS)
    largura, altura = imagem.size
    xs = list(range(0, max(1, largura - TAM + 1), PASSO)); ys = list(range(0, max(1, altura - TAM + 1), PASSO))
    if xs[-1] != largura - TAM: xs.append(max(0, largura - TAM))
    if ys[-1] != altura - TAM: ys.append(max(0, altura - TAM))
    caixas, notas = [], []
    for y in ys:
        for x in xs:
            recorte = imagem.crop((x, y, x + TAM, y + TAM))
            for r in modelo.predict(recorte, imgsz=TAM, conf=confianca, verbose=False)[0].boxes:
                x0, y0, x1, y1 = r.xyxy[0].tolist()
                caixas.append([x0 + x, y0 + y, x1 + x, y1 + y]); notas.append(float(r.conf[0]))
    if caixas:
        manter = torchvision.ops.nms(torch.tensor(caixas), torch.tensor(notas), 0.5)
        caixas = [caixas[i] for i in manter]; notas = [notas[i] for i in manter]
    return imagem, caixas, notas

enviados = files.upload()
pathlib.Path('/content/previsoes').mkdir(exist_ok=True)
for nome in enviados:
    imagem, caixas, notas = prever_em_recortes(melhor, PImage.open(nome))
    desenho = ImageDraw.Draw(imagem)
    for (x0, y0, x1, y1), nota in zip(caixas, notas):
        desenho.rectangle([x0, y0, x1, y1], outline=(255, 60, 40), width=5)
    saida = '/content/previsoes/' + pathlib.Path(nome).stem + '_previsao.jpg'
    imagem.save(saida, quality=90)
    print(nome, ':', len(caixas), 'caixas')
shutil.make_archive('/content/previsoes', 'zip', '/content/previsoes')
files.download('/content/previsoes.zip')